# Zero-shot football vision

**The idea.** Take a public YOLO model that has never seen our footage and never been trained on it.
Let it *propose* regions (people, maybe a ball) and let ByteTrack *follow* them. Then ask the OpenAI
**Decisions API** to *judge* each region into exactly four labels:

`ball` · `referee` · `team_1_player` · `team_2_player`

No hand labels. No training. No fine-tuning. Every answer is cached, counted and capped.

**What we promise to say plainly**

- YOLO's own labels are metadata. They never become a class. Decisions is the classifier.
- Confidence numbers are **not probabilities**. The thresholds are experimental, not calibrated.
- `team_1` / `team_2` are **arbitrary visual groups** (ordered by who appears first), not clubs or home/away.
- When the model had to choose, the row says `forced=True`. We keep that flag.
- **Nothing is interpolated.** A frame we did not search is `unsearched`, not empty.

**The steps**

| | |
|---|---|
| 0 | setup, cost cap, cache |
| 1 | load three sample frames |
| 2 | find the on-screen HUD |
| 3 | YOLO proposals, kept only if the feet are on the pitch |
| 4 | guess the kits from colour |
| 5 | project the cost (zero calls) |
| 6 | Decisions validates the kits |
| 7 | Decisions labels each region, then is forced to pick one of four |
| 8 | the ball search |
| 9 | remove duplicates, look at the frames |
| 10 | tracking with ByteTrack |
| 11 | export and honest numbers |
| 12 | limits |

By default the notebook makes **no API calls**: the sample answers are replayed from a cache.

## Step 0. Setup

**Cost: the sample frames are free, your own images are not.**
Every Decisions answer is saved in `samples/decisions_cache.json`. The same question later is answered from that file: no call, no cost.

- **3 sample frames:** already cached. No API key, no cost, same result every time.
- **Your own images:** new crops, so new calls. Run a dry run first (`LIVE_CALLS = False`) to count them. Then put `OPENAI_API_KEY` in `.env` and set `LIVE_CALLS = True`. `MAX_CALLS` is a hard cap. Re-running the same images is free.
- To use your own frames, put PNGs in `samples/frames/` named like `frame_01_src950.png`, `frame_02_src1250.png`, ... (the number after `src` is the source frame index).

The shipped cache is **copied** into `out/` first, so your runs never change `samples/decisions_cache.json` unless you set `WRITE_BACK_TO_SAMPLES = True` at the very end.

In [ ]:
import os, sys, json, shutil
from pathlib import Path
from collections import Counter

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, "src")          # the pipeline modules live in src/

import cv2, numpy as np, matplotlib.pyplot as plt
import config_io as C
import detect as D
import decide as DC
import track as T
from config_io import Config

LIVE_CALLS = False                 # False: replay the cache and count what is missing; nothing is sent
MAX_CALLS = 70                     # hard stop: no more than this many live calls
LIVE_CLIP_CALLS = False            # also needs LIVE_CALLS: True lets the tracking step (10) spend extra calls on the sample clip
WRITE_BACK_TO_SAMPLES = False      # True: copy new answers back into samples/decisions_cache.json
YOLO_IMGSZ = 1280                  # YOLO input size for proposals; 0 = ultralytics default (640). 1280 for 1280x720 frames with 30-70 px players
HUD_MODE = "none"                  # "auto" finds a HUD; "none" masks nothing (no overlay, or a fixed camera: SoccerTrack v2)

OUT = Path(os.environ.get("ZSFV_OUT", ROOT / "out"))        # git-ignored scratch folder
IMG = ROOT / "docs" / "img"                                  # fixed figures that the docs reuse
IMG.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# the cache is copied, never edited in place
if not (OUT / "decisions_cache.json").exists():
    shutil.copy(ROOT / "samples" / "decisions_cache.json", OUT / "decisions_cache.json")

# public COCO weights. If models/yolo11s.pt is missing, ultralytics downloads "yolo11s.pt" by name.
WEIGHTS = "models/yolo11s.pt" if Path("models/yolo11s.pt").is_file() else "yolo11s.pt"

cfg = Config(
    video=str(ROOT / "samples" / "frames"), out_dir=str(OUT), yolo_weights=WEIGHTS,
    yolo_imgsz=YOLO_IMGSZ, hud_mode=HUD_MODE,
    n_frames=3, max_ball_regions_per_frame=48,
    max_api_calls=MAX_CALLS, dry_run=not LIVE_CALLS,
)

class CountingClient(C.DecisionClient):
    # Same client, plus one tally of where every answer came from: cache, live, or dry_run (skipped).
    def __init__(self, cfg):
        super().__init__(cfg)
        self.sources = Counter()
    def ask(self, content, questions, tag):
        res = super().ask(content, questions, tag)
        self.sources[res["source"]] += 1
        return res

def show(img, title="", w=11, save=None):
    # Show a BGR image. With save="NN_name" also write docs/img/NN_name.jpg (<=1280px wide, <200KB).
    if save:
        out = img if img.shape[1] <= 1280 else cv2.resize(img, (1280, int(img.shape[0] * 1280 / img.shape[1])), interpolation=cv2.INTER_AREA)
        for q in (88, 80, 72, 64, 56, 48):
            ok, buf = cv2.imencode(".jpg", out, [cv2.IMWRITE_JPEG_QUALITY, q])
            if len(buf) < 195_000:
                break
        (IMG / f"{save}.jpg").write_bytes(buf.tobytes())
    plt.figure(figsize=(w, img.shape[0] / img.shape[1] * w))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis("off")
    if title: plt.title(title, fontsize=10)
    plt.show()

print("python", sys.version.split()[0], "| openai", __import__("openai").__version__, "| model", C.DECISIONS_MODEL)
print("live calls allowed:", LIVE_CALLS, "| cap:", MAX_CALLS, "| API key found:", bool(C.load_key()))
print("weights:", WEIGHTS)

## Step 1. Three real frames

These are three 1280x720 windows cut from a SoccerTrack v2 panorama (CC BY 4.0), saved as lossless PNGs. The file name carries the
position in the original video (`frame_02_src1250.png` is source frame 1250). We build the same record
that `config_io.extract_frames` returns, so every later step works the same, but the repo needs no big video.

In [ ]:
import re
FPS = 25.0                                    # the source clip's frame rate
frames = []
for p in sorted((ROOT / "samples" / "frames").glob("frame_*_src*.png")):
    img = cv2.imread(str(p))
    src = int(re.search(r"_src(\d+)\.png$", p.name).group(1))
    frames.append({"frame_id": f"f{int(re.match(r'frame_(\d+)_', p.name).group(1))}", "source_index": src, "timestamp_s": round(src / FPS, 4),
                   "width": img.shape[1], "height": img.shape[0], "file": str(p), "image": img})
for f in frames:
    print(f"{f['frame_id']}  source frame {f['source_index']:>4}  t={f['timestamp_s']:5.2f}s  {f['width']}x{f['height']}")
show(DC.contact_sheet([f["image"] for f in frames], cols=3, width=420), "the three source frames", save="01_sample_frames")

## Step 2. Where is the HUD?

On broadcast video the scoreboard and minimap sit on top of the picture (SoccerTrack v2 has none, so this notebook sets `HUD_MODE = "none"`). The minimap is full of little white dots that
look like balls, so we mask it out.

The trick: HUD edges land on the **same pixels in every frame**, while real scene edges move with
the camera. Keep the edges that never move, near the borders, and turn each cluster into a rectangle.
This is automatic, and it is cautious: if the mask would swallow more than 30% of the frame, it is thrown away.

In [ ]:
mask, hud = C.estimate_hud_mask(frames, mode=HUD_MODE)
print("method:", hud["method"], "| playable fraction:", hud["playable_fraction"])
for p in hud["panels"]:
    print("  panel", p["bbox"], "fill", p["fill"])
vis = frames[0]["image"].copy()
vis[mask == 0] = (0, 0, 255)
show(cv2.addWeighted(frames[0]["image"], 0.5, vis, 0.5, 0), "red = ignored by the search", save="02_hud_mask")

## Step 3. YOLO proposes, the pitch filters

YOLO is public and pretrained. We keep its **boxes** and throw away its opinions: `yolo_label` is
carried along only so we can later ask whether Decisions disagreed.

A box must be **standing on the pitch**. We test the *feet* (bottom-centre point), not the whole box,
because heads rise above the grass. The tolerance grows sideways and downward only. Growing it
upward reaches into the stands and lets spectators in.

In [ ]:
proposals = D.propose_objects(cfg, frames, mask)
print("proposals kept:", len(proposals), "| YOLO labels (metadata only):", dict(Counter(p["yolo_label"] for p in proposals)))
for f in frames:
    print(f"  {f['frame_id']}: off-pitch rejected = {f['proposals_off_pitch']}")

f = frames[0]
pm = D.pitch_mask(f["image"], mask); mg = cfg.pitch_margin_px
grown = cv2.dilate(pm, np.ones((mg + 1, 2 * mg + 1), np.uint8), anchor=(mg, mg))
vis = f["image"].copy()
vis[grown == 0] = (vis[grown == 0] * 0.25).astype("uint8")
cnts, _ = cv2.findContours(grown, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(vis, cnts, -1, (0, 255, 255), 2)
for p in [p for p in proposals if p["frame_id"] == f["frame_id"]]:
    x0, y0, x1, y1 = p["box"]
    cv2.rectangle(vis, (x0, y0), (x1, y1), (0, 220, 0), 1)
    cv2.circle(vis, ((x0 + x1) // 2, y1 - 2), 3, (255, 255, 0), -1)
show(vis, "green = kept, yellow line = pitch boundary, cyan dot = the feet point we test", save="03_proposals")

## Step 4. Kit colours (a guess, not an answer)

Take the torso of each box, remove grass pixels, drop blurry crops, then cluster the median colours with k-means.
This gives **clusters, not teams**. Colour alone cannot tell a goalkeeper from a referee from a team,
so the clusters go to Decisions for judgement in step 6.

In [ ]:
hyp = D.kit_hypotheses(frames, proposals)
print("usable torso crops:", hyp.get("usable_crops"), "| ok:", hyp["ok"])
tiles = []
for g in hyp["groups"]:
    print(f"  {g['group_id']}  n={g['size']:<3} Lab={g['lab_center']}")
    m = D.montage(frames, g["members"]); cv2.putText(m, g["group_id"], (4, 14), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    tiles.append(m)
hh = max(t.shape[0] for t in tiles)
strip = np.hstack([np.pad(t, ((0, hh - t.shape[0]), (0, 8), (0, 0))) for t in tiles])
show(strip, "one row of crops per cluster: a hypothesis only", w=11, save="04_kit_clusters")

## Step 5. What would this cost?

Zero calls. We count what we are about to ask for and check it against the cap.
Cached answers cost nothing, so a replay shows `live_calls: 0`.

In [ ]:
dc = CountingClient(cfg)
planned = {"kit validation": len(hyp["groups"]), "kit pair checks (worst case)": max(len(hyp["groups"]) - 1, 0), "object calls": len(proposals),
           "ball search windows": cfg.max_ball_regions_per_frame * len(frames)}
for k, v in planned.items():
    print(f"{k:20} {v}")
print("optimistic total    ", sum(planned.values()), "| cache holds", len(dc.cache), "answers | cap", cfg.max_api_calls)
print("a cache miss with LIVE_CALLS=False is skipped, never sent.")

## Step 6. Decisions judges the kit clusters

For each cluster we send a montage of its sharpest crops and ask two questions:
is it really **one kit**? and is it an **outfield team, goalkeeper, official or a mix**?
The best outfield cluster becomes a team. Every other validated cluster gets one same-or-different question against it. A cluster judged the same kit is recorded as merged and not used. Of the clusters judged a **different kit**, the second team is the one with the cleanest reference (highest `same_kit_p` x role confidence, ties to the larger cluster), not the biggest; the others are recorded as `different_kit_not_chosen`. The two teams are then ordered by earliest appearance.

**These are arbitrary visual groups.** Not clubs, not home/away.

If two clusters cannot be validated, player labels stay `uncertain`. We never invent a team.

In [ ]:
cfg.dry_run = not LIVE_CALLS
dc = CountingClient(cfg)
kits = D.validate_kits(cfg, dc, frames, hyp)
print("resolved:", kits["resolved"], "|", kits.get("reason", ""))
for g in kits["groups"]:
    print(f"  {g['group_id']}: same_kit_p={g['same_kit_p']} role={g['role']} conf={g['role_conf']} n={g['size']} src={g['source']}" + (f" -> merged into {g['merged_into']} (same kit)" if g.get('merged_into') else "") + (f" -> {g['outcome']}" if g.get('outcome') else ""))
for c in kits.get("pair_checks", []):
    print(f"  pair {c['a']} vs {c['b']}: different_p={c['different_p']} different={c['different']} src={c['source']}")
if kits["resolved"]:
    print("mapping:", kits["mapping"]); print("rule   :", kits["rule"])
    show(kits["board"], "reference board: TEAM 1 (left) | TEAM 2 (right)", w=7, save="06_kit_board")
step6 = dict(dc.sources); print("step 6 answer sources:", step6)

## Step 7. Label every region, then force a choice

**First pass.** Each request carries three images: the TEAM 1 reference, the TEAM 2 reference,
then the crop. Two separate labelled images worked; one montage captioned "left/right" did not,
because the model never reliably read the spatial caption. Below the accept threshold, a box stays
`uncertain` and the reason is recorded.

In [ ]:
obj_dets = DC.classify_objects(cfg, dc, frames, proposals, kits)
print("first pass:", dict(Counter(d["cls"] for d in obj_dets)))
print("uncertain because:", dict(Counter(d["uncertain_reason"] for d in obj_dets if d["cls"] == "uncertain")))
print("YOLO said person, Decisions said background or ball:",
      sum(1 for d in obj_dets if d["yolo_label"] == "person" and d["cls"] in ("background", "ball")))

**Second pass on the undecided.** A six-way question splits probability, so the right class lands near 0.45 and never
clears the bar. We re-ask only the undecided ones with **four labels**, a bigger crop and a referee reference.
Same threshold; we do not lower the bar. `pass1_confidence` is kept so the upgrade can be audited.

In [ ]:
obj_dets, pass2 = DC.resolve_uncertain(cfg, dc, frames, obj_dets, kits)
print("pass 2:", json.dumps(pass2)); print("now:", dict(Counter(d["cls"] for d in obj_dets)))

**Force four labels.** Anything still undecided must pick one. Two guards remain: things judged *not a person*
(banners, crowd) are **dropped**, and `ball` is **not offered** here, because it came from person boxes.
The argmax is accepted whatever its confidence, so each such row is tagged `forced=True`.
Many forced labels are low confidence and some are wrong. Goalkeepers are the clearest case.

In [ ]:
obj_dets, forced_stats = DC.force_labels(cfg, dc, frames, obj_dets, kits)
print("forced:", json.dumps(forced_stats))
for d in [x for x in obj_dets if x.get("forced")]:
    flag = "  <- below the accept threshold, treat as a guess" if (d["confidence"] or 0) < cfg.t_object_accept else ""
    print(f"  {d['proposal_id']:<10} {d['cls']:<14} {d['confidence']:.2f}{flag}")
f = frames[0]
fid = f["frame_id"]
show(DC.render(f, [d for d in obj_dets if d["frame_id"] == f["frame_id"]]),
     f"{fid} after step 7: orange = team_1, yellow = team_2, magenta = referee, * = forced", save="07_labels_f0")
step7 = dict(dc.sources); print("answer sources so far (steps 6-7):", step7)

## Step 8. The ball

The hard part: the ball is about 5 pixels and white on green.

1. Cover the **pitch only** with overlapping windows. The stands add only false positives.
2. Ask `ball_present` per window. Windows are ranked by a small-bright-blob cue and spread out.
3. **Image processing**, not the model, turns a positive window into a tight box. The window verdict never gives coordinates.
4. A separate `is_ball` question confirms the tight crop.

If presence holds but no tight box is confirmed, the result is `ball_present_unlocalized`. Windows we never reached are logged `unsearched`.
Coverage matters more than cleverness: run the table below to see it.

In [ ]:
print(f"{'windows/frame':>14} {'pitch covered':>14}")
for budget in (12, 24, 48, 96):
    probe = Config(**{**vars(cfg), "max_ball_regions_per_frame": budget})
    covs = []
    for f in frames:
        regs, _ = DC.ball_regions(probe, f, mask)
        c = np.zeros(mask.shape, np.uint8)
        for r in regs:
            b = r["box"]; c[b[1]:b[3], b[0]:b[2]] = 1
        pmk = D.pitch_mask(f["image"], mask); covs.append(100 * (c[pmk > 0] > 0).mean())
    print(f"{budget:>14} {np.mean(covs):>13.1f}%" + ("   <- used" if budget == cfg.max_ball_regions_per_frame else ""))

f = frames[0]
pmk = D.pitch_mask(f["image"], mask); regs, omitted = DC.ball_regions(cfg, f, mask)
vis = f["image"].copy(); vis[pmk == 0] = (vis[pmk == 0] * 0.35).astype("uint8")
for r in regs:
    x0, y0, x1, y1 = r["box"]; cv2.rectangle(vis, (x0, y0), (x1, y1), (0, 200, 255), 1)
show(vis, f"{f['frame_id']}: {len(regs)} search windows on the pitch", save="08_ball_search")

In [ ]:
ball_dets, regions = DC.find_ball(cfg, dc, frames, mask)
print("window verdicts:", dict(Counter(r["verdict"] for r in regions)))
for d in ball_dets:
    print(f"  {d['frame_id']} {d['localization']:<26} conf={d['confidence']} box={d['box']}")
print("answer sources, all steps:", dict(dc.sources), "|", dc.stats())

## Step 9. Remove duplicates, look at the frames

Duplicate removal is **class-aware**: two boxes on one player collapse, but a ball on top of a player survives.

How to read the pictures: **thick box** = tight and confirmed. **Thin box with `?`** = ball present but not localized.
**`*`** = forced label. **Dashed green** = a search window that said "ball here", which is a region, not a box.

In [ ]:
dets, suppressed = D.nms_class_aware(obj_dets + ball_dets)
print(f"kept {len(dets)}, suppressed {len(suppressed)} duplicates |", dict(Counter(d["cls"] for d in dets)))
rendered = [DC.render(f, [d for d in dets if d["frame_id"] == f["frame_id"]],
                      [r for r in regions if r["frame_id"] == f["frame_id"]]) for f in frames]
show(DC.contact_sheet(rendered, cols=3, width=420), "all three frames", w=15, save="09_contact_sheet")

## Step 10. Tracking: identity across a clip

A YOLO box has no memory. ByteTrack gives each person an **id that persists across frames**, so we can
judge a *track* instead of every box in every frame. This runs on a 3 second clip and makes **zero API calls**.

In [ ]:
CLIP = ROOT / "samples" / "clip" / "clip_3s.mp4"
res = T.run_clip(str(CLIP), 0.0, 3.0, str(OUT / "clip"), dry_run=True, yolo_weights=WEIGHTS, imgsz=YOLO_IMGSZ, hud_mode=HUD_MODE)   # dry_run: no client is even created
proj = res["projection"]; print(json.dumps(proj))

In [ ]:
o = T.Opts(yolo_weights=WEIGHTS, kit_samples=2, imgsz=YOLO_IMGSZ, hud_mode=HUD_MODE)
fps, first, n = T._window(str(CLIP), 0.0, 3.0)
hud_c, _ = T.hud_mask(str(CLIP), first, n, o.hud_samples, o.hud_mode)
tr = T.track_clip(str(CLIP), 0.0, 3.0, cfg, hud_c, o)
rng = np.random.default_rng(3)
colour = {}
cap = cv2.VideoCapture(str(CLIP)); tiles = []
for rel in range(tr["n_frames"]):
    ok, img = cap.read()
    if rel not in (0, 25, 50, 74): continue
    for b in tr["frames"][rel]["boxes"]:
        c = colour.setdefault(b["track_id"], tuple(int(v) for v in rng.integers(60, 255, 3)))
        x0, y0, x1, y1 = b["box"]
        cv2.rectangle(img, (x0, y0), (x1, y1), c, 2 if b["on_pitch"] else 1)
        cv2.putText(img, f"#{b['track_id']}", (x0, max(y0 - 4, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, c, 1, cv2.LINE_AA)
    tiles.append(img)
cap.release()
show(DC.contact_sheet(tiles, cols=2, width=640), "ByteTrack ids at 0.0s, 1.0s, 2.0s and 3.0s (thin box = feet off the pitch)", w=12, save="10_tracking")
qualified, skipped = T.qualify_tracks(tr["tracks"], o)
print(f"{len(tr['tracks'])} track ids, {len(qualified)} qualify for a vote, {len(skipped)} skipped:")
for tid, why in list(skipped.items())[:6]: print(f"   #{tid}: {why}")

**What a live per-track vote adds.** For each qualifying track we pick about three crops from different moments of its life
and ask Decisions **once**. Early, middle and late crops are chosen on purpose: if the tracker swapped two players,
the crops disagree, and the track is **split in time** instead of getting one wrong label. The vote is then painted onto
every frame where the track appears.

In [ ]:
per_frame_calls = sum(1 for fr in tr["frames"] for b in fr["boxes"] if b["on_pitch"])
print(f"judging every on-pitch box in every frame: {per_frame_calls} calls")
print(f"one vote per track (about 3 crops each):   {proj['total_calls_est']} calls, estimated, ball search included")
if qualified:
    t0 = max(qualified, key=lambda t: t["n_frames"])
    crops = T.pick_crops(t0, o.crops_per_track)
    hh = 160; row = [cv2.resize(c["crop"], (int(c["crop"].shape[1] * hh / c["crop"].shape[0]), hh)) for c in crops]
    show(np.hstack(row), f"track #{t0['track_id']}: the crops one vote would see (rel. frames {[c['rel'] for c in crops]})", w=6)

if LIVE_CALLS and LIVE_CLIP_CALLS:
    # Spends calls on this clip, capped by MAX_CALLS. The sample cache does not hold these answers.
    live = T.run_clip(str(CLIP), 0.0, 3.0, str(OUT / "clip_live"), dry_run=False, yolo_weights=WEIGHTS, max_calls=MAX_CALLS, imgsz=YOLO_IMGSZ, hud_mode=HUD_MODE)
    print("live calls:", live["api_calls"], "| cache hits:", live["cache_hits"], "| tracks:", live["track_stats"])
else:
    print("No per-track vote was requested. Set LIVE_CALLS and LIVE_CLIP_CALLS to True (and mind MAX_CALLS) to see one.")

## Step 11. Export and the honest numbers

The export keeps what you need to audit a claim later: frame ids, boxes in original pixels, decision type,
class, confidence, uncertainty reasons, localization status and the call log.

In [ ]:
meta = DC.export(cfg, frames, dets, regions, kits, hud, dc, suppressed)
cls = Counter(d["cls"] for d in dets)
forced = [d for d in dets if d.get("forced")]
print("classes :", dict(cls))
print("forced  :", len(forced), "| below accept threshold:", sum(1 for d in forced if (d["confidence"] or 0) < cfg.t_object_accept))
print("balls   :", cls["ball"], "boxes, of which tight:", sum(1 for d in dets if d["cls"] == "ball" and d["localization"] == "tight"),
      "| frames with a ball box:", len({d['frame_id'] for d in dets if d['cls'] == 'ball'}), "of", len(frames))
print("calls   : live", dc.calls, "| cache hits", dc.cache_hits, "| answer sources", dict(dc.sources))
EXPECTED = {"team_1_player": 37, "team_2_player": 28, "ball": 0, "referee": 4}     # the published set (f1, f2, f4: 69 boxes, 21 forced; the one false ball box in f2 removed). A 3-frame re-run builds new kit clusters, so a cache miss is possible: it does not have to match
print("matches the earlier run:", all(cls[k] == v for k, v in EXPECTED.items()), "| expected", EXPECTED)
for p in ("results.json", "detections.csv", "regions.csv"): print("wrote", cfg.path(p))
if WRITE_BACK_TO_SAMPLES:
    dc.save(); shutil.copy(OUT / "decisions_cache.json", ROOT / "samples" / "decisions_cache.json"); print("cache copied back to samples/")

## Step 12. What this does and does not show

Read the **pictures**, not only the counts.

- **Four labels only.** `*` rows were forced. Filter on `forced=True` for the cautious view.
- **Not probabilities.** Confidences and thresholds are experimental. The same ball scored very differently
  in different windows; overlapping coverage matters more than any one call.
- **Teams are visual groups.** Not clubs. Order is by earliest appearance.
- **Goalkeepers and referees in odd kits** are the weak spot. A keeper matches neither outfield kit, so the model picks a side on thin evidence.
- **The ball is the hardest label.** It is small, blurred and easy to confuse with socks or shirts. A confirmed box is not proof; re-confirmation alone did not catch every false seed.
- **Nothing is interpolated.** Between confirmed ball boxes we draw nothing. Unsearched windows are logged as such.
- **No ground truth.** The references are the model's own answers plus eyeballing. No accuracy figure is claimed,
  and this cannot support possession, passes or any 2D pitch position (pitch placement is out of scope).
- **Three frames and one clip** are a demonstration, not an evaluation. Change the footage and run again.

A result that survives footage you did not tune on is worth something. This notebook only shows the method works end to end.